Copyright 2026 Google LLC.

SPDX-License-Identifier: Apache-2.0


<a href="https://colab.research.google.com/github/datacommonsorg/api-python/blob/master/notebooks/analyzing_obesity_prevalence_sdmx.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


**Notebook Version** - 2.0.0 (SDMX port)


# Case Study: Predicting Obesity Prevalence in US Counties

**Objective:** This notebook demonstrates how to use Data Commons SDMX APIs to build a linear regression model predicting the prevalence of obesity in US counties.

**Background:** Obesity prevalence is known to correlate with various health and socioeconomic factors [[1]](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC3198075/)[[2]](https://www.ncbi.nlm.nih.gov/pubmed/26562758). Data for these factors often reside in separate datasets from different government agencies:

- The Centers for Disease Control (CDC) provides health-condition prevalence data, such as obesity and high blood pressure.
- The US Bureau of Labor Statistics (BLS) provides unemployment rates.
- The US Census Bureau provides poverty data and population counts.

Data Commons aggregates these diverse datasets into a unified knowledge graph, simplifying data access and analysis.

**Approach:** This notebook uses Data Commons to retrieve 2021 observations for the following variables for US counties:

- [Percentage of Adult Population That Is Obese](https://datacommons.org/tools/statvar#sv=Percent_Person_Obesity) (CDC)—target variable.
- [Percentage of Adult Population With High Blood Pressure](https://datacommons.org/tools/statvar#sv=Percent_Person_WithHighBloodPressure) (CDC)—predictor variable.
- [Unemployment Rate of a Population](https://datacommons.org/tools/statvar#sv=UnemploymentRate_Person) (BLS)—predictor variable.
- [Population Below Poverty Level Status in Past Year](https://datacommons.org/tools/statvar#sv=Count_Person_BelowPovertyLevelInThePast12Months) (Census)—used to calculate poverty rate.
- [Total Population](https://datacommons.org/tools/statvar#sv=Count_Person) (Census)—used to calculate poverty rate.

A linear regression model will be trained using high blood pressure prevalence, unemployment rate, and the calculated poverty rate to predict obesity prevalence.

*Note:* The US Census also provides unemployment statistics. Using BLS data here is for demonstration purposes. Comparing results using Census unemployment data could be a potential extension.


## 1. Set up environment

### 1.1. Install libraries

Install pySDMX for SDMX metadata and the Data Commons Python client for place names.


In [ ]:
!pip install "pysdmx[data,json]==1.19.0" "datacommons-client==2.1.6" --quiet


### 1.2. Configure the API key

Obtain your own Data Commons API key from [apikeys.datacommons.org](https://apikeys.datacommons.org/) and replace the placeholder in the cell below with your key.

The notebook sends this key in the `X-Api-Key` header for SDMX requests and configures the Data Commons Python client with the same key. After changing the key, rerun this cell and the data cells below; the installation, imports, and helper definitions do not need to be rerun.


In [ ]:
from datacommons_client import DataCommonsClient

DC_API_KEY = "Replace this string with your API key"
dc_client = DataCommonsClient(api_key=DC_API_KEY)


### 1.3. Import dependencies

Import the libraries and define the helpers required for data manipulation, modeling, and plotting.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

import io
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
import requests
from pysdmx.io import read_sdmx

API_ROOT = "https://api.datacommons.org"
SDMX_DATA_URL = f"{API_ROOT}/sdmx/v3/data/dataflow/DC/DF_OBS/1.0.0/*"
SDMX_AVAILABILITY_URL = (
    f"{API_ROOT}/sdmx/v3/availability/dataflow/DC/DF_OBS/1.0.0/*"
)


def _headers(accept=None):
    if not DC_API_KEY:
        raise ValueError("Set DC_API_KEY before requesting Data Commons.")
    headers = {"X-Api-Key": DC_API_KEY}
    if accept:
        headers["Accept"] = accept
    return headers


def sdmx_data(constraints):
    response = requests.get(
        SDMX_DATA_URL,
        params=constraints,
        headers=_headers("application/vnd.sdmx.data+csv;version=2.0.0"),
        timeout=120,
    )
    response.raise_for_status()
    frame = pd.read_csv(io.BytesIO(response.content), low_memory=False)
    frame["OBS_VALUE"] = pd.to_numeric(frame["OBS_VALUE"], errors="coerce")
    return frame


def availability_values(component, constraints):
    response = requests.get(
        f"{SDMX_AVAILABILITY_URL}/{component}",
        params=constraints,
        headers=_headers(),
        timeout=120,
    )
    response.raise_for_status()
    message = read_sdmx(io.BytesIO(response.content))
    if not message.structures:
        raise ValueError("The SDMX Availability request returned no constraint.")
    return {
        value.value
        for region in message.structures[0].cube_regions
        for key_value in region.key_values
        if key_value.id == component
        for value in key_value.values
    }


def require_variables(label, variables, constraints):
    constraints = {
        key: value for key, value in constraints.items()
        if not (key == "c[TIME_PERIOD]" and value == "LATEST")
    }
    available = availability_values(
        "variableMeasured",
        {**constraints, "c[variableMeasured]": ",".join(variables)},
    )
    missing = sorted(set(variables) - available)
    if missing:
        raise ValueError(f"{label} is missing variables: {', '.join(missing)}")
    print(f"{label}: all {len(variables)} variables are available.")


# The client follows response pagination within each input batch.
def _fetch_name_batch(nodes):
    return {
        dcid: name.value
        for dcid, name in dc_client.node.fetch_entity_names(
            entity_dcids=nodes
        ).items()
    }


def fetch_node_names(dcids, batch_size=500):
    unique_dcids = list(dict.fromkeys(dcids))
    if not unique_dcids:
        return {}
    batches = [
        unique_dcids[index:index + batch_size]
        for index in range(0, len(unique_dcids), batch_size)
    ]
    names = {dcid: dcid for dcid in unique_dcids}
    with ThreadPoolExecutor(max_workers=min(8, len(batches))) as executor:
        for batch_names in executor.map(_fetch_name_batch, batches):
            names.update(batch_names)
    return names


## 2. Data acquisition

Fetch statistical observations for the specified variables for all US counties in 2021 using SDMX Data.

The variables come from three source facets:

- CDC500 with `AgeAdjustedPrevalence` for obesity and high blood pressure.
- BLS LAUS with `BLSSeasonallyUnadjusted` for unemployment.
- Census ACS 5-year with `CensusACS5yrSurvey` for poverty counts and total population.

SDMX Availability first confirms that the expected variables are available under each selected facet. The three SDMX Data requests then run in parallel. The Data Commons Python client supplies county names for the DCIDs returned by SDMX.


In [ ]:
SOURCE_GROUPS = [
    {
        "label": "CDC health",
        "variables": [
            "Percent_Person_Obesity",
            "Percent_Person_WithHighBloodPressure",
        ],
        "provenance": "dc/base/CDC500",
        "measurement_method": "AgeAdjustedPrevalence",
    },
    {
        "label": "BLS unemployment",
        "variables": ["UnemploymentRate_Person"],
        "provenance": "dc/base/BLS_LAUS",
        "measurement_method": "BLSSeasonallyUnadjusted",
    },
    {
        "label": "Census poverty and population",
        "variables": [
            "Count_Person_BelowPovertyLevelInThePast12Months",
            "Count_Person",
        ],
        "provenance": "dc/base/CensusACS5YearSurvey",
        "measurement_method": "CensusACS5yrSurvey",
    },
]


def group_constraints(group):
    return {
        "c[observationAbout.containedInPlace+]": "country/USA",
        "c[observationAbout.typeOf]": "County",
        "c[provenance]": group["provenance"],
        "c[measurementMethod]": group["measurement_method"],
        "c[TIME_PERIOD]": "2021",
    }


for group in SOURCE_GROUPS:
    require_variables(
        group["label"],
        group["variables"],
        group_constraints(group),
    )


def fetch_group(group):
    return sdmx_data({
        **group_constraints(group),
        "c[variableMeasured]": ",".join(group["variables"]),
    })


with ThreadPoolExecutor(max_workers=3) as executor:
    group_frames = list(executor.map(fetch_group, SOURCE_GROUPS))

observations = pd.concat(group_frames, ignore_index=True)
observations = observations.dropna(subset=["OBS_VALUE"])
names = fetch_node_names(observations["observationAbout"].unique())

us_county_observations_df = (
    observations.rename(columns={
        "observationAbout": "entity",
        "variableMeasured": "variable",
        "OBS_VALUE": "value",
    })
    .assign(entity_name=lambda frame: frame["entity"].map(names))
)
us_county_observations_df.head(5)


## 3. Data preparation

Process the fetched data for modeling:

1. **Select source facets:** The SDMX requests already constrain each group to its relevant provenance and measurement method.
2. **Select columns:** Keep only the essential columns: `entity`, `entity_name`, `variable`, and `value`.
3. **Pivot:** Reshape the DataFrame so each variable becomes a column, indexed by county `entity` and `entity_name`.
4. **Calculate poverty rate:** Compute the poverty-rate percentage using the population count and the count of people below the poverty level.
5. **Handle missing values:** Drop counties with any missing value among the selected variables.


In [ ]:
filtered_df = us_county_observations_df[
    ["entity", "entity_name", "variable", "value"]
]

pivoted_df = filtered_df.pivot_table(
    index=["entity", "entity_name"],
    columns="variable",
    values="value",
)
pivoted_df = pivoted_df[pivoted_df["Count_Person"] > 0]
pivoted_df["PovertyRate"] = (
    pivoted_df["Count_Person_BelowPovertyLevelInThePast12Months"]
    / pivoted_df["Count_Person"]
    * 100
)
pivoted_df.dropna(inplace=True)
print(f"Complete counties: {len(pivoted_df):,}")
pivoted_df.head(5)


## 4. Exploratory data analysis

Visualize the relationships between the target variable (obesity prevalence) and the predictor variables (high blood pressure prevalence, unemployment rate, and poverty rate) using scatter plots. This helps assess potential correlations.


In [ ]:
pivoted_df.plot(
    kind="scatter",
    x="Percent_Person_Obesity",
    y="Percent_Person_WithHighBloodPressure",
    grid=True,
    figsize=(10, 6),
    title="Obesity vs. High Blood Pressure in US Counties",
)
plt.xlabel("Obesity Prevalence (Age-Adjusted %)")
plt.ylabel("High Blood Pressure Prevalence (Age-Adjusted %)")
plt.show()

pivoted_df.plot(
    kind="scatter",
    x="UnemploymentRate_Person",
    y="Percent_Person_Obesity",
    grid=True,
    figsize=(10, 6),
    title="Unemployment Rate vs. Obesity Prevalence in US Counties",
)
plt.xlabel("Unemployment Rate")
plt.ylabel("Obesity Prevalence (Age-Adjusted %)")
plt.show()

pivoted_df.plot(
    kind="scatter",
    x="PovertyRate",
    y="Percent_Person_Obesity",
    grid=True,
    figsize=(10, 6),
    title="Poverty Rate vs. Obesity Prevalence in US Counties",
)
plt.xlabel("Poverty Rate")
plt.ylabel("Obesity Prevalence (Age-Adjusted %)")
plt.show()


*Observation:* The scatter plots suggest positive correlations between obesity prevalence and each of the predictor variables.


## 5. Model training

Train a linear regression model to predict obesity prevalence based on the selected predictors.

The model follows the form:

$$f_\theta(x) = \theta_0 + \theta_1 (\text{high blood pressure}) + \theta_2 (\text{unemployment}) + \theta_3 (\text{poverty rate})$$

### 5.1. Prepare features and target variable

Define the feature matrix `X` (predictors) and the target vector `Y` (obesity prevalence).

### 5.2. Split data

Split the data into training and testing sets: 80% training and 20% testing. A fixed random state makes repeated runs comparable.

### 5.3. Train linear regression model

Instantiate and train scikit-learn's [LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html) model using the training data.


In [ ]:
features = [
    "Percent_Person_WithHighBloodPressure",
    "UnemploymentRate_Person",
    "PovertyRate",
]
target = "Percent_Person_Obesity"

x_train, x_test, y_train, y_test = train_test_split(
    pivoted_df[features],
    pivoted_df[[target]],
    test_size=0.2,
    random_state=42,
)

model = LinearRegression(fit_intercept=True)
model.fit(x_train, y_train)

print(f"Training set size: {len(x_train):,} samples")
print(f"Test set size: {len(x_test):,} samples")
print(f"Model intercept: {model.intercept_}")
print(f"Model coefficients: {model.coef_}")


## 6. Model evaluation

Assess the performance of the trained model using Mean Squared Error (MSE) and residual analysis.

### 6.1. Calculate Mean Squared Error

Define a function for MSE and calculate it for both the training and test sets. Lower MSE indicates a better fit.

### 6.2. Analyze residuals

Calculate and plot the residual proportion, `(predicted - actual) / actual`, for the test set. Residuals ideally should be scattered around zero.


In [ ]:
def mse(y_true, y_pred):
    """Compute mean squared error."""
    return np.mean((y_pred - y_true) ** 2)


train_pred = model.predict(x_train)
test_pred = model.predict(x_test)
train_mse = mse(y_train.to_numpy(), train_pred)
test_mse = mse(y_test.to_numpy(), test_pred)

print(f"Training MSE: {train_mse:.4f}")
print(f"Test MSE: {test_mse:.4f}")

residual_proportion = (
    (test_pred.ravel() - y_test[target].to_numpy())
    / y_test[target].to_numpy()
)
plt.figure(figsize=(10, 6))
plt.title("Residual Proportions Plot")
plt.xlabel("Test Data Row Index")
plt.ylabel("residual / actual obesity prevalence")
plt.scatter(range(len(residual_proportion)), residual_proportion)
plt.show()


The printed MSE values compare model fit on the training and test sets. The residual plot shows the direction and relative size of prediction errors across test counties.


## 7. Conclusion and next steps

This notebook demonstrated the use of Data Commons SDMX APIs to acquire data from multiple sources—CDC, BLS, and Census—and build a simple linear regression model to predict obesity prevalence in US counties. Data Commons streamlines the data gathering and integration process.

The resulting model, using high blood pressure prevalence, unemployment rate, and poverty rate, provides a baseline prediction.

**Potential improvements and further exploration:**

- **Add more variables:** Include other variables known or hypothesized to correlate with obesity, such as:
  - `Percent_Person_WithHighCholesterol`
  - `Percent_Person_WithDiabetes`
  - educational attainment levels
  - access to healthy food outlets
  - physical inactivity rates
- **Feature engineering:** Create new features from existing ones.
- **Model selection:** Experiment with different regression models, such as Ridge, Lasso, or tree-based models.
- **Geographic analysis:** Explore spatial patterns in obesity prevalence and model errors.
- **Alternative data sources:** Compare model performance using Census unemployment data instead of BLS data.

Data Commons provides access to a wide range of variables, enabling exploration of correlations with university counts, crime rates, or environmental factors, potentially leading to more comprehensive models.
